In [ ]:
# One-time check: find how far the stage must move off the start switch to release it
switch_var = "_LF" if START_END_SIGN > 0 else "_LR"
galil_seek_end(START_END_SIGN, HOME_SPEED)
on_state = gcmd(f"MG {switch_var}{GALIL_AXIS}")
moved = 0
while gcmd(f"MG {switch_var}{GALIL_AXIS}") == on_state and moved < 20_000:
    galil_move(-START_END_SIGN * 100, 5_000)
    moved += 100
print(f"Switch reads {on_state} when active, released after {moved} counts ({moved / COUNTS_PER_MM:.3f} mm)")
galil_go_to_start()

In [ ]:
# One-time check: count the full travel to identify the stage scale
galil_seek_end(-START_END_SIGN, HOME_SPEED)
full_travel = abs(galil_state()[0]) + HOME_BACKOFF
print(f"Full travel: {full_travel} counts")
galil_go_to_start()

# V-MD3 SAR Data Acquisition

Headless continuous-motion SAR data collection for the V-MD3 radar with a Galil-driven CDS-3310 stage. Each sweep records RADC / RFFT / DONE frames to a `.bin` file while the stage moves, with the stage position logged for every frame.

**One-time install:** install gclib from Galil's site (the .exe), then in PowerShell run `py -m pip install https://www.galil.com/sw/pub/python/gclib-1.0.1-py3-none-any.whl`

**Before running:** close GDK, connect the radar and the Galil, then run Cell 0 (approve the UAC prompt). Use `python measure.py` separately for the live heatmap.

**Workflow:**

- **Cell 0** - Radar network setup (re-run after replugging the radar).
- **Cell 1** - Imports and helper functions (run once).
- **Cell 2** - User configuration.
- **Cell 3** - Connect to the radar.
- **Cell 3B** - Connect to the Galil and home the stage.
- **Cell 3C** - Measure the frame period and sweep speed.
- **Cell 4** - One continuous sweep (repeat per sweep).
- **Cell 5** - Disconnect radar and Galil.

## Cell 0 - Radar network setup

Runs `setup_radar.ps1` (same folder as this notebook) as administrator. A UAC prompt appears unless Jupyter is already elevated. The script output prints here when it finishes.

In [ ]:
import os
import base64
import ctypes
import subprocess
from pathlib import Path

# Setup script location and a temp log the elevated process writes to
SETUP_PS1 = Path("C:/My Computer/vmd3/setup_radar.ps1").resolve()
SETUP_LOG = Path(os.environ["TEMP"]) / "vmd3_setup_log.txt"


def run_radar_setup(ps1=SETUP_PS1, log=SETUP_LOG):
    """Run setup_radar.ps1 as administrator and print its output here."""
    if not ps1.is_file():
        raise FileNotFoundError(f"Setup script not found: {ps1}")
    log.unlink(missing_ok=True)

    # Build a PowerShell command that runs the script and saves all output to the log
    ps1_q = str(ps1).replace("'", "''")
    log_q = str(log).replace("'", "''")
    inner = f"& '{ps1_q}' *>&1 | Out-File -FilePath '{log_q}' -Encoding utf8"
    encoded = base64.b64encode(inner.encode("utf-16-le")).decode()
    ps_args = ["-NoProfile", "-ExecutionPolicy", "Bypass", "-EncodedCommand", encoded]

    # Run directly if already elevated, otherwise go through a UAC prompt
    if ctypes.windll.shell32.IsUserAnAdmin():
        subprocess.run(["powershell", *ps_args])
    else:
        print("[cell 0] Approve the UAC prompt to run the radar setup...")
        arg_str = " ".join(ps_args)
        launcher = f"Start-Process powershell -Verb RunAs -Wait -WindowStyle Hidden -ArgumentList '{arg_str}'"
        subprocess.run(["powershell", "-NoProfile", "-Command", launcher])

    # Show the script output and confirm it passed
    if not log.is_file():
        raise RuntimeError("No setup output. The UAC prompt may have been declined.")
    text = log.read_text(encoding="utf-8-sig")
    print(text)
    if "All checks passed" not in text:
        raise RuntimeError("Radar setup did not pass. See the messages above.")
    print("[cell 0] Radar network ready.")


run_radar_setup()

## Cell 1 - Imports and helper functions

In [ ]:
import os
import csv
import time
import socket
import numpy as np
import pandas as pd
import gclib
from datetime import datetime
from lib.vmd3 import VMD3, RdotConfig

# Map user-facing stream names to RdotConfig flags and UDP headers
STREAM_MAP = {
    'radc': RdotConfig.RADC,
    'rfft': RdotConfig.RFFT,
    'done': RdotConfig.DONE,
}
HEADER_MAP = {
    'radc': b'RADC',
    'rfft': b'RFFT',
    'done': b'DONE',
}
CYCLE_ORDER = ["radc", "rfft", "done"]   # order the streams arrive within one radar cycle


def read_any_frame(vmd3, headers):
    """Read the next UDP frame whose header is in `headers`. Returns full frame."""
    while True:
        data = vmd3.recv_udp()
        if data[0:4] not in headers:
            continue
        resp_len = int.from_bytes(data[4:8], byteorder='little')
        while len(data) < resp_len + 8:
            data += vmd3.recv_udp()
        if resp_len != len(data[8:]):
            continue
        return data


def drain_udp(vmd3):
    """Throw away everything already sitting in the UDP buffer."""
    old_timeout = vmd3.sockUDP.gettimeout()
    vmd3.sockUDP.settimeout(0.05)
    try:
        while True:
            try:
                vmd3.recv_udp()
            except socket.timeout:
                break
    finally:
        vmd3.sockUDP.settimeout(old_timeout)


def wait_for_frame(vmd3, header):
    """Block until the next complete frame with this header arrives."""
    while True:
        try:
            return read_any_frame(vmd3, (header,))
        except socket.timeout:
            continue


def sync_stream():
    """Last selected stream in each radar cycle, used to mark cycle boundaries."""
    return [s for s in CYCLE_ORDER if s in STREAMS][-1]


def galil_reconnect(attempts=3):
    """Close and reopen the Galil connection to clear out-of-sync replies."""
    try:
        galil.GClose()
    except gclib.GclibError:
        pass
    for i in range(attempts):
        time.sleep(0.5)
        try:
            galil.GOpen(GALIL_ADDRESS)
            return
        except gclib.GclibError:
            if i == attempts - 1:
                raise


def gcmd(cmd):
    """Send one Galil command, reconnecting once after a timeout, and return its reply."""
    try:
        return galil.GCommand(cmd).strip()
    except gclib.GclibError as e:
        if "timed out" not in str(e):
            raise RuntimeError(f"Galil rejected '{cmd}': {galil.GCommand('TC1').strip()}") from e
        print(f"[galil] '{cmd}' timed out, reconnecting...")
        galil_reconnect()
        return galil.GCommand(cmd).strip()


def galil_state():
    """Return (position in counts, True if the axis is still moving)."""
    pos = float(gcmd(f"MG _TP{GALIL_AXIS}"))
    moving = float(gcmd(f"MG _BG{GALIL_AXIS}")) > 0.5
    return int(pos), moving


def galil_wait(timeout_s=60):
    """Poll until the axis stops moving, stopping it if it takes too long."""
    t_end = time.perf_counter() + timeout_s
    while galil_state()[1]:
        if time.perf_counter() > t_end:
            gcmd(f"ST{GALIL_AXIS}")
            raise TimeoutError("Stage move did not finish in time.")
        time.sleep(0.05)


def galil_move(counts, speed, absolute=False, timeout_s=60):
    """Move the stage at a given speed and block until the move finishes."""
    gcmd(f"SP{GALIL_AXIS}={int(speed)}")
    gcmd(f"{'PA' if absolute else 'PR'}{GALIL_AXIS}={int(counts)}")
    gcmd(f"BG{GALIL_AXIS}")
    try:
        galil_wait(timeout_s)
    except BaseException:
        gcmd(f"ST{GALIL_AXIS}")
        raise


def galil_seek_end(direction, speed, timeout_s=60):
    """Jog toward an end stop and stop once the stage stops advancing."""
    gcmd(f"JG{GALIL_AXIS}={int(direction * speed)}")
    try:
        gcmd(f"BG{GALIL_AXIS}")
    except RuntimeError as e:
        if "limit switch" in str(e):
            return   # already sitting on the start-end limit switch
        raise
    t_end = time.perf_counter() + timeout_s
    last_pos, last_t = galil_state()[0], time.perf_counter()
    try:
        while True:
            time.sleep(0.05)
            pos, moving = galil_state()
            if not moving:
                break   # the limit switch stopped it
            if abs(pos - last_pos) > 100:
                last_pos, last_t = pos, time.perf_counter()
            elif time.perf_counter() - last_t > 0.3:
                break   # pressed against the end stop
            if time.perf_counter() > t_end:
                raise TimeoutError("Stage never reached the end stop.")
    finally:
        gcmd(f"ST{GALIL_AXIS}")
        galil_wait(10)


def galil_home():
    """Drive into the start end stop, back off, and call that position zero."""
    gcmd(f"SH{GALIL_AXIS}")
    gcmd(f"AC{GALIL_AXIS}={SWEEP_ACCEL}")
    gcmd(f"DC{GALIL_AXIS}={SWEEP_ACCEL}")
    galil_seek_end(START_END_SIGN, HOME_SPEED)
    gcmd(f"SH{GALIL_AXIS}")   # re-enable in case the end stop tripped the error limit
    galil_move(-START_END_SIGN * HOME_BACKOFF, HOME_SPEED)
    time.sleep(0.2)
    gcmd(f"DP{GALIL_AXIS}=0")


def galil_go_to_start():
    """Return to the zero position set by galil_home and let it settle."""
    galil_move(0, HOME_SPEED, absolute=True)
    time.sleep(0.2)


print('[cell 1] Imports and helpers loaded.')

## Cell 2 - Configuration

Edit values below for the session, then run the cell. Re-run it any time something is changed.

- **Changing `SESSION_NAME` or sweep settings** only needs this cell re-run, then go to Cell 4.
- **Changing `RSET_CONFIG` or `STREAMS`** needs a reconnect: re-run this cell, then Cell 3 and Cell 3C.

In [ ]:
# ─── Where to save binary data ───
OUTPUT_BASE = "data"   # base directory for all sessions
SESSION_NAME = "continuous_motion_test"   # folder under OUTPUT_BASE

# ─── What to record ───
STREAMS = ["radc"]   # any of: "radc", "rfft", "done"

# ─── Radar mode ───
RSET_CONFIG = 0   # 0 = 2D, 6 m max range, 4.69 cm range bin

# ─── Connection (no need change) ───
RADAR_IP = "192.168.100.201"
UDP_TIMEOUT = 1.0   # seconds

# ─── Continuous sweep (Cell 4) ───
SWEEP_LENGTH_MM = 100.0
N_SWEEP_FRAMES = 101
FRAME_PERIOD_OVERRIDE_S = None   # None uses the Cell 3C measurement, or set e.g. 0.131
SWEEP_ACCEL = 1_000_000   # counts/s^2, reaches sweep speed in under 0.1 s
SWEEP_BASENAME = "sweep"

# ─── Galil stage (CDS-3310) ───
GALIL_ADDRESS = "COM4 --baud 115200 -d"   # same port GDK uses, direct connection
GALIL_AXIS = "A"
COUNTS_PER_MM = 8_000   # 1 count = 125 nm
STAGE_RANGE_COUNTS = 809_954
START_END_SIGN = +1   # +1 starts at the end the old PR +1100000 drove to, -1 for the other end
HOME_BACKOFF = 1_000   # 0.125 mm off the end switch
HOME_SPEED = 50_000   # counts/s for homing and returning to start

# ─── Validate & normalize configurations (no need change) ───
STREAMS = [s.strip().lower() for s in STREAMS]
STREAMS = list(dict.fromkeys(STREAMS))   # remove duplicates, keep order

_bad = [s for s in STREAMS if s not in STREAM_MAP]
if _bad:
    raise ValueError(f"Unknown stream(s): {_bad}. Valid options: radc, rfft, done")
if not STREAMS:
    raise ValueError("STREAMS is empty. Select at least one of: radc, rfft, done")
if not (0 <= RSET_CONFIG <= 8):
    raise ValueError("RSET_CONFIG must be 0-8")

SWEEP_COUNTS = int(round(SWEEP_LENGTH_MM * COUNTS_PER_MM))
if HOME_BACKOFF + SWEEP_COUNTS > STAGE_RANGE_COUNTS:
    raise ValueError("Backoff plus sweep length exceeds the stage travel")
if N_SWEEP_FRAMES < 2:
    raise ValueError("N_SWEEP_FRAMES must be at least 2")

# Session paths
SESSION_DIR = os.path.join(OUTPUT_BASE, SESSION_NAME)
BINS_DIR = os.path.join(SESSION_DIR, "bins")
CSV_PATH = os.path.join(SESSION_DIR, "scan_log.csv")

# Compute RDOT flag list from the selected streams
SELECTED_RDOT = [STREAM_MAP[s] for s in STREAMS]

# Summary
print(f"[cell 2] Session folder : {SESSION_DIR}")
print(f"[cell 2] Streams        : {STREAMS}  (RDOT = 0x{sum(r.value for r in SELECTED_RDOT):02X})")
print(f"[cell 2] RSET mode      : {RSET_CONFIG}")
print(f"[cell 2] Sweep          : {SWEEP_LENGTH_MM:.1f} mm, target {N_SWEEP_FRAMES} frames, sync on {sync_stream().upper()}")

## Cell 3 - Connect to the radar

Run once at the start of a session. This opens the connection, configures the radar with the RSET mode and selected streams, and creates the session folder.

Safe to re-run: if a connection is already open, it closes it first. Re-run this (then Cell 3C) after changing `RSET_CONFIG` or `STREAMS` in Cell 2.

The connection stays open until you run Cell 5 (or restart the kernel), so you **don't** need to re-run this between sweeps.

In [ ]:
# Close any connection left open from a previous run so re-running doesn't leave a dangling socket
if 'vmd3' in globals() and vmd3 is not None:
    try:
        vmd3.sockTCP.settimeout(2.0)
        vmd3.disconnect()
        print("[cell 3] Closed existing connection.")
    except Exception as e:
        print(f"[cell 3] Warning closing old connection: {e}")
        for s in (vmd3.sockTCP, vmd3.sockUDP):
            try:
                s.close()
            except Exception:
                pass
    vmd3 = None
    time.sleep(2)   # give the radar a moment to release the old session

# ─── Create the session folders + CSV (committed here, not in Cell 2) ───
os.makedirs(BINS_DIR, exist_ok=True)
if not os.path.exists(CSV_PATH):
    with open(CSV_PATH, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([
            "filename", "scan_pos_cm", "timestamp_start", "timestamp_stop",
            "duration_s", "n_frames", "rset_mode", "streams", "notes",
        ])
    print(f"[cell 3] Created {CSV_PATH}")
else:
    print(f"[cell 3] Appending to existing {CSV_PATH}")


def connect_radar(attempts=3):
    """Open and configure the V-MD3, retrying if the handshake stalls."""
    for i in range(attempts):
        radar = VMD3(tcp_ip=RADAR_IP)
        try:
            radar.connect()
            radar.sockTCP.settimeout(5.0)
            radar.set_rset_config(RSET_CONFIG)
            radar.set_output_config(SELECTED_RDOT)
            return radar
        except OSError as e:
            print(f"[cell 3] Handshake stalled ({e or 'timeout'}), retrying ({i + 1}/{attempts})...")
            for s in (getattr(radar, "sockTCP", None), getattr(radar, "sockUDP", None)):
                try:
                    s.close()
                except Exception:
                    pass
            time.sleep(2)
    raise RuntimeError("Radar did not respond. Power cycle it, wait ~10 s, then re-run Cell 0 and Cell 3.")


# ─── Connect ───
print(f"[cell 3] Connecting to V-MD3 at {RADAR_IP}...")
vmd3 = connect_radar()
vmd3.sockUDP.settimeout(UDP_TIMEOUT)
vmd3.sockUDP.setsockopt(socket.SOL_SOCKET, socket.SO_RCVBUF, 8 * 1024 * 1024)   # buffers frames while Python queries the Galil

print(f"[cell 3] Ready. RSET={RSET_CONFIG}, streams={STREAMS}, "
      f"RDOT=0x{sum(r.value for r in SELECTED_RDOT):02X}")

## Cell 3B - Connect to the Galil and home the stage

Close GDK first. Run once per session. Drives into the start end stop, backs off 1 mm, and zeros the position there. Re-run if the stage is bumped or power cycled.

In [ ]:
# Close any previous Galil handle so re-running doesn't leave a stale connection
if 'galil' in globals() and galil is not None:
    try:
        galil.GClose()
    except Exception:
        pass

# Connect and stop anything left running from GDK
galil = gclib.py()
galil.GOpen(GALIL_ADDRESS)
print(f"[cell 3B] Connected: {galil.GInfo()}")
gcmd(f"ST{GALIL_AXIS}")
galil_wait()
try:
    gcmd("HX")   # halt any program left running from GDK
except RuntimeError:
    pass

# Home to the start end
print("[cell 3B] Homing...")
galil_home()
print(f"[cell 3B] Homed. Position {galil_state()[0]} counts = start of sweep.")

## Cell 3C - Measure the frame period

Times 30 radar cycles with the current streams and computes the sweep speed for `N_SWEEP_FRAMES` over `SWEEP_LENGTH_MM`. Re-run after changing STREAMS or RSET.

In [ ]:
N_TIMING_CYCLES = 30
WAVELENGTH_MM = 299_792_458 / 61.0e9 * 1e3


def measure_frame_period(n_cycles=N_TIMING_CYCLES):
    """Time n radar cycles and return the median cycle period in seconds."""
    header = HEADER_MAP[sync_stream()]
    drain_udp(vmd3)
    stamps = []
    while len(stamps) < n_cycles + 1:
        wait_for_frame(vmd3, header)
        stamps.append(time.perf_counter())
    return float(np.median(np.diff(stamps)))


# Measure and report the resulting sweep plan
MEASURED_FRAME_PERIOD_S = measure_frame_period()
speed = SWEEP_COUNTS / ((N_SWEEP_FRAMES - 1) * MEASURED_FRAME_PERIOD_S)
spacing_mm = SWEEP_LENGTH_MM / (N_SWEEP_FRAMES - 1)
print(f"[cell 3C] Frame period  : {MEASURED_FRAME_PERIOD_S*1e3:.1f} ms ({1/MEASURED_FRAME_PERIOD_S:.2f} Hz)")
print(f"[cell 3C] Sweep speed   : {speed:.0f} counts/s ({speed/COUNTS_PER_MM:.2f} mm/s)")
print(f"[cell 3C] Sweep time    : {SWEEP_COUNTS/speed:.1f} s for {N_SWEEP_FRAMES} frames")
print(f"[cell 3C] Frame spacing : {spacing_mm:.3f} mm (λ/4 = {WAVELENGTH_MM/4:.3f} mm)")

# Warn if the spacing is too coarse for alias-free SAR
if spacing_mm > WAVELENGTH_MM / 4:
    min_frames = int(np.ceil(SWEEP_LENGTH_MM / (WAVELENGTH_MM / 4))) + 1
    print(f"[cell 3C] Warning: spacing exceeds λ/4. Use N_SWEEP_FRAMES ≥ {min_frames} to avoid grating lobes.")

## Cell 4 - One continuous sweep

Returns the stage to start, waits for a radar cycle boundary, starts the sweep, and records until the stage stops and that cycle finishes. The speed is set so the sweep spans `N_SWEEP_FRAMES` frames (set in Cell 2, measured period from Cell 3C).

Saves `sweep_###.bin` plus `sweep_###_positions.csv` with one row per record in file order:

- `stream`, `stream_frame` - which frame in the `.bin` (e.g. RADC frame 12)
- `t_s` - arrival time since motion start
- `galil_counts` - raw encoder position
- `x_mm` - 0 at the start (right) end to `SWEEP_LENGTH_MM` at the far end
- `x_center_mm` - position from the sweep center, + at the right end, − at the left end
- `moving` - 1 if the stage was moving

Use `x_center_mm` (or `x_mm`) for SAR processing rather than assuming uniform spacing. Repeat for each sweep.

In [ ]:
SWEEP_NAME = "test_sweep_closer_to_limit_switch"   # saved as ball_sweep.bin, or ball_sweep_2.bin, _3... if the name is taken
SWEEP_NOTES = ""   # optional note for scan_log.csv


def next_sweep_path():
    """Path for SWEEP_NAME in BINS_DIR, adding _2, _3... instead of overwriting."""
    path = os.path.join(BINS_DIR, f"{SWEEP_NAME}.bin")
    i = 2
    while os.path.exists(path):
        path = os.path.join(BINS_DIR, f"{SWEEP_NAME}_{i}.bin")
        i += 1
    return path

def record_sweep(notes=""):
    """Run one continuous sweep and record every frame while the stage moves."""
    period = FRAME_PERIOD_OVERRIDE_S or globals().get("MEASURED_FRAME_PERIOD_S")
    if period is None:
        raise RuntimeError("No frame period. Run Cell 3C or set FRAME_PERIOD_OVERRIDE_S in Cell 2.")
    speed = int(round(SWEEP_COUNTS / ((N_SWEEP_FRAMES - 1) * period)))
    timeout_s = SWEEP_COUNTS / speed + 10
    half_mm = SWEEP_LENGTH_MM / 2

    # File paths and per-record bookkeeping
    filepath = next_sweep_path()
    filename = os.path.basename(filepath)
    pos_csv = filepath.replace(".bin", "_positions.csv")
    accepted = tuple(HEADER_MAP[s] for s in STREAMS)
    header_to_name = {HEADER_MAP[s]: s for s in STREAMS}
    sync_name = sync_stream()
    per_stream = {s: 0 for s in STREAMS}
    rows = []
    bytes_written = 0

    # Park at the start and load the sweep profile without starting it
    print("[cell 4] Moving to start...")
    galil_go_to_start()
    gcmd(f"SP{GALIL_AXIS}={speed}")
    gcmd(f"AC{GALIL_AXIS}={SWEEP_ACCEL}")
    gcmd(f"DC{GALIL_AXIS}={SWEEP_ACCEL}")
    gcmd(f"PR{GALIL_AXIS}={-START_END_SIGN * SWEEP_COUNTS}")

    # Wait for the end of a radar cycle so motion starts on a cycle boundary
    drain_udp(vmd3)
    wait_for_frame(vmd3, HEADER_MAP[sync_name])

    # Start the motion and the clock together
    gcmd(f"BG{GALIL_AXIS}")
    t0 = time.perf_counter()
    start_time = datetime.now()
    print(f"[cell 4] Sweeping {SWEEP_LENGTH_MM:.0f} mm at {speed/COUNTS_PER_MM:.2f} mm/s "
          f"(target {N_SWEEP_FRAMES} frames) → {filename}")

    # Record until the stage has stopped and the current cycle is complete
    out = open(filepath, "wb")
    try:
        while True:
            if time.perf_counter() - t0 > timeout_s:
                raise TimeoutError("Sweep took too long. Check the radar stream and the stage.")
            try:
                frame = read_any_frame(vmd3, accepted)
            except socket.timeout:
                continue
            t_frame = time.perf_counter() - t0
            pos, moving = galil_state()
            name = header_to_name[frame[0:4]]

            out.write(frame)
            bytes_written += len(frame)
            x_mm = -START_END_SIGN * pos / COUNTS_PER_MM
            x_center_mm = half_mm - x_mm
            rows.append([len(rows), name, per_stream[name], f"{t_frame:.4f}", pos,
                         f"{x_mm:.4f}", f"{x_center_mm:.4f}", int(moving)])
            per_stream[name] += 1

            if name == sync_name:
                print(f"   cycle {per_stream[name]:3d}   x = {x_mm:7.2f} mm   "
                      f"center {x_center_mm:+7.2f} mm   {'moving' if moving else 'stopped'}")
                if not moving:
                    break
    except BaseException:
        gcmd(f"ST{GALIL_AXIS}")
        raise
    finally:
        out.close()

    stop_time = datetime.now()
    duration = (stop_time - start_time).total_seconds()

    # Save the per-record position log
    with open(pos_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["record", "stream", "stream_frame", "t_s", "galil_counts",
                         "x_mm", "x_center_mm", "moving"])
        writer.writerows(rows)

    # Add a summary row to the session log
    with open(CSV_PATH, "a", newline="") as f:
        csv.writer(f).writerow([
            filename, "sweep",
            start_time.isoformat(timespec="seconds"),
            stop_time.isoformat(timespec="seconds"),
            f"{duration:.2f}", len(rows), RSET_CONFIG, "+".join(STREAMS),
            f"{notes} | {SWEEP_LENGTH_MM:g} mm at {speed} cts/s, target {N_SWEEP_FRAMES} frames",
        ])

    # Summarize the aperture using the first stream of each cycle
    first = [s for s in CYCLE_ORDER if s in STREAMS][0]
    xc = np.array([float(r[6]) for r in rows if r[1] == first])
    print(f"[cell 4] Saved {filename}: {len(rows)} records, {bytes_written/1e6:.2f} MB, {duration:.1f} s")
    print(f"[cell 4] {len(xc)} {first.upper()} frames (target {N_SWEEP_FRAMES}), "
          f"center position {xc[0]:+.2f} to {xc[-1]:+.2f} mm, mean spacing {np.mean(np.abs(np.diff(xc))):.3f} mm")
    # Warn about dropped frames, which show up as a gap in position
    gaps = np.abs(np.diff(xc))
    big = np.where(gaps > 1.5 * SWEEP_LENGTH_MM / (N_SWEEP_FRAMES - 1))[0]
    if big.size:
        print(f"[cell 4] Warning: {big.size} dropped frame gap(s), largest {gaps.max():.2f} mm "
              f"near center {xc[big[0]]:+.2f} mm. Consider re-running this sweep.")
    print(f"[cell 4] Positions → {os.path.basename(pos_csv)}")


record_sweep(SWEEP_NOTES)

## Cell 5 - Disconnect

Run at the end of a session to stop the stage and close both the radar and Galil connections cleanly.

Safe to run even if nothing is connected since it just reports that there's nothing to close.

In [ ]:
if 'vmd3' in globals() and vmd3 is not None:
    try:
        vmd3.sockTCP.settimeout(2.0)
        vmd3.disconnect()
        print("[cell 5] Disconnected from V-MD3.")
    except Exception as e:
        print(f"[cell 5] Disconnect warning: {e}")
        # Force-close the sockets so the OS releases the ports
        try:
            vmd3.sockTCP.close()
        except Exception:
            pass
        try:
            vmd3.sockUDP.close()
        except Exception:
            pass
        print("[cell 5] Sockets force-closed.")
    finally:
        vmd3 = None
else:
    print("[cell 5] Nothing to disconnect (no open connection).")

# Stop the stage and close the Galil connection
if 'galil' in globals() and galil is not None:
    try:
        gcmd(f"ST{GALIL_AXIS}")
        galil.GClose()
        print("[cell 5] Disconnected from Galil.")
    except Exception as e:
        print(f"[cell 5] Galil close warning: {e}")
    finally:
        galil = None
else:
    print("[cell 5] No Galil connection to close.")

## Cell 6 - Convert .bin to long-format CSV

Decodes a saved `.bin` file into a lossless "long" CSV: one row per (frame, bin, chirp, channel) value, with complex I/Q split into separate `I` and `Q` columns.

Set `BIN_TO_CONVERT` to the sweep file you want, choose which streams to export, then run. The CSV is saved next to the `.bin` with `_long.csv` added.

Warning: long format is large. One RADC or RFFT frame is 32,768 rows, so a ~101-frame sweep with RADC + RFFT is about 6.6 million rows. Lossless but big.

In [ ]:
# ─── What to convert ───
BIN_TO_CONVERT = os.path.join(BINS_DIR, "sweep_001.bin")   # path to a .bin file
CONVERT_STREAMS = ["radc", "rfft"]   # which streams to export: radc and/or rfft
CONVERT_MODE = "2D"   # "2D" or "3D" (matches how it was recorded)

# ─── Where to save ───
OUTPUT_CSV_PATH = os.path.splitext(BIN_TO_CONVERT)[0] + "_long.csv"

# ─── Decoders ───
def _get_frames(filepath, header, expected_len):
    """Extract all valid payloads for a given header from a .bin file."""
    with open(filepath, "rb") as f:
        raw = f.read()
    frames, start = [], 0
    while True:
        idx = raw.find(header, start)
        if idx == -1:
            break
        if idx + 8 > len(raw):
            break
        payload_len = int.from_bytes(raw[idx+4:idx+8], byteorder="little")
        if payload_len == expected_len:
            p0, p1 = idx + 8, idx + 8 + expected_len
            if p1 <= len(raw):
                payload = raw[p0:p1]
                if header not in payload:   # reject header-in-payload false hits
                    frames.append(payload)
        start = idx + 1
    return frames


def _decode_radc_2d(frame):
    """Decode one 2D RADC payload to (samples, chirps, channels)."""
    raw = np.frombuffer(frame, dtype="<i2").reshape(64, 4, 256)
    q = raw[:, :, 0::2].astype(np.float64)
    i = raw[:, :, 1::2].astype(np.float64)
    complex_data = i + 1j * q
    return np.transpose(complex_data, (2, 0, 1))


def _decode_rfft_2d(frame):
    """Decode one 2D RFFT payload to (range_bin, chirp, ch)."""
    raw = np.frombuffer(frame, dtype="<i2").reshape(4, 128, 64, 2)
    i = raw[:, :, :, 1].astype(np.float64)
    q = raw[:, :, :, 0].astype(np.float64)
    complex_data = i + 1j * q
    return np.transpose(complex_data, (1, 2, 0))


# ─── Build the long dataframe ───
def bin_to_long_df(filepath, streams, mode):
    """Decode the chosen streams of a .bin file into one long dataframe."""
    expected_len = 131072 if mode == "2D" else 196608
    header_map = {"radc": b"RADC", "rfft": b"RFFT"}
    all_rows = []

    for s in streams:
        header = header_map[s]
        frames = _get_frames(filepath, header, expected_len)
        if not frames:
            print(f"  [{s}] no frames found, skipping")
            continue

        for f_idx, frame in enumerate(frames):
            cube = _decode_radc_2d(frame) if s == "radc" else _decode_rfft_2d(frame)

            # Flatten the (bin, chirp, channel) cube into index columns
            n_bin, n_chirp, n_chan = cube.shape
            bb, cc, hh = np.meshgrid(
                np.arange(n_bin), np.arange(n_chirp), np.arange(n_chan),
                indexing="ij"
            )
            df = pd.DataFrame({
                "file": os.path.basename(filepath),
                "stream": s,
                "frame": f_idx,
                "bin": bb.ravel(),   # sample index (radc) or range bin (rfft)
                "chirp": cc.ravel(),
                "channel": hh.ravel(),
                "I": np.real(cube).ravel(),
                "Q": np.imag(cube).ravel(),
            })
            all_rows.append(df)
            print(f"  [{s}] frame {f_idx}: {len(df)} rows")

    if not all_rows:
        print("No data converted.")
        return None
    return pd.concat(all_rows, ignore_index=True)


# ─── Run ───
print(f"[cell 6] Converting {os.path.basename(BIN_TO_CONVERT)} "
      f"(streams={CONVERT_STREAMS}, mode={CONVERT_MODE})...")

df = bin_to_long_df(BIN_TO_CONVERT, CONVERT_STREAMS, CONVERT_MODE)

if df is not None:
    # Write the CSV
    out_dir = os.path.dirname(OUTPUT_CSV_PATH)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)
    df.to_csv(OUTPUT_CSV_PATH, index=False)
    print(f"[cell 6] Wrote {OUTPUT_CSV_PATH}")
    print(f"[cell 6] Shape: {df.shape[0]} rows × {df.shape[1]} cols")

    # Per-stream row breakdown
    stream_counts = df["stream"].value_counts()
    for s in CONVERT_STREAMS:
        print(f"[cell 6]   {s}: {int(stream_counts.get(s, 0))} rows")
    print(f"[cell 6]   total: {len(df)} rows")
    print(df.head())